# ChordSense Pro — Model v4 Frame-Level Training on Colab
GPU: L4 24GB | Runtime: GPU

> **Luôn chạy cell Setup đầu tiên** sau mỗi lần restart runtime.
> Đảm bảo 3 file `chords.parquet` + `classes.json` mới (đã cứu `bass_id` Hooktheory ~13.7% slash chords) đã được upload đè lên `MyDrive/ChordSensePro/DE/data/export/v3/`.

## 0. Setup — chạy đầu tiên sau mỗi restart

In [ ]:
import os, shutil, yaml
from pathlib import Path

# ── Mount Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Config ───────────────────────────────────────────────────
DRIVE_PROJECT = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR      = "/content/ChordSensePro"
DATA_V2       = "/content/data_v2"
DATA_DST      = "/content/data_v3"
P1_RUN_NAME   = "v4_5s_p1"
P2_RUN_NAME   = "v4_5s_p2"

# ── Clone / pull repo ─────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
print("cwd:", os.getcwd())

# ── Install deps ──────────────────────────────────────────────
!pip install -q -r requirements.txt

# ── Build P1 config ───────────────────────────────────────────
cfg_p1 = yaml.safe_load(Path(f"configs/{P1_RUN_NAME}.yaml").read_text())
cfg_p1["data_dir"]       = DATA_DST
cfg_p1["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", P1_RUN_NAME)
cfg_p1["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", P1_RUN_NAME, "results.csv")
os.makedirs(cfg_p1["checkpoint_dir"], exist_ok=True)
colab_cfg = f"configs/colab_{P1_RUN_NAME}.yaml"
Path(colab_cfg).write_text(yaml.dump(cfg_p1, default_flow_style=False))

# ── Build P2 config ───────────────────────────────────────────
cfg_p2 = yaml.safe_load(Path(f"configs/{P2_RUN_NAME}.yaml").read_text())
cfg_p2["data_dir"]       = DATA_DST
cfg_p2["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME)
cfg_p2["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME, "results.csv")
os.makedirs(cfg_p2["checkpoint_dir"], exist_ok=True)
colab_p2_cfg = f"configs/colab_{P2_RUN_NAME}.yaml"
Path(colab_p2_cfg).write_text(yaml.dump(cfg_p2, default_flow_style=False))

P1_BEST       = os.path.join(cfg_p1["checkpoint_dir"], "best.pt")
P2_BEST       = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")
OUT_ONNX      = os.path.join(DRIVE_PROJECT, "AI", "results", f"model_{P2_RUN_NAME}.onnx")
OUT_EVAL_JSON = os.path.join(DRIVE_PROJECT, "AI", "results", P2_RUN_NAME, "test_metrics.json")

print("P1 checkpoint_dir:", cfg_p1["checkpoint_dir"])
print("P2 checkpoint_dir:", cfg_p2["checkpoint_dir"])
print("P1 best.pt exists :", os.path.exists(P1_BEST))
print("P2 best.pt exists :", os.path.exists(P2_BEST))
print("\nSetup done. Run cells below in order.")


In [ ]:
!nvidia-smi

## 1. Prepare Data v3 (Sync Audio + Always Refresh Parquet/Classes)
Copy audio từ Drive về `/content/data_v3` (nếu chưa có) và luôn đồng bộ mới nhất `chords.parquet` + `classes.json`.

In [ ]:
def _audio_ok(d):
    p = os.path.join(d, "train", "audio")
    return os.path.isdir(p) and len(os.listdir(p)) > 0

DATA_V3_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v3")
DATA_V2_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v2")

if _audio_ok(DATA_DST):
    print("v3 audio already cached at", DATA_DST)
elif _audio_ok(DATA_V3_SRC):
    print("Copying v3 from Drive...")
    !rm -rf {DATA_DST}
    !cp -r {DATA_V3_SRC} {DATA_DST}
else:
    print("v3 audio missing on Drive — rebuilding from v2")
    if not os.path.exists(DATA_V2):
        print("Copying v2 from Drive...")
        !cp -r {DATA_V2_SRC} {DATA_V2}
    !python ../DE/scripts/export_dataset_v3.py --v2-dir {DATA_V2} --out {DATA_DST}

# Luôn cập nhật metadata & parquet mới nhất từ Drive (chỉ ~3.5 MB)
if os.path.isdir(DATA_V3_SRC):
    for fn in ("classes.json", "config.yaml"):
        src_f = os.path.join(DATA_V3_SRC, fn)
        if os.path.exists(src_f):
            shutil.copy2(src_f, os.path.join(DATA_DST, fn))
    for sp in ("train", "val", "test"):
        src_pq = os.path.join(DATA_V3_SRC, sp, "chords.parquet")
        dst_pq = os.path.join(DATA_DST, sp, "chords.parquet")
        if os.path.exists(src_pq):
            os.makedirs(os.path.dirname(dst_pq), exist_ok=True)
            shutil.copy2(src_pq, dst_pq)

import pandas as pd
df = pd.read_parquet(os.path.join(DATA_DST, "train", "chords.parquet"))
n_audio = len(os.listdir(os.path.join(DATA_DST, "train", "audio")))
slash_pct = 100.0 * (df["bass_id"] != 12).mean() if "bass_id" in df.columns else 0.0
print(f"train rows: {len(df):,} | slash chords: {slash_pct:.2f}% (expected ~13.66%)")
print("train audio files:", n_audio)
assert slash_pct > 10.0, "CẢNH BÁO: chords.parquet trên Drive vẫn là bản cũ thiếu inversion Hooktheory!"


---
## Phase 1 — Train Frame-Level Head + Temporal Context (MERT frozen)

In [ ]:
!python -m src.train --config {colab_cfg}

In [ ]:
# Resume Phase 1 nếu bị disconnect
P1_LAST = os.path.join(cfg_p1["checkpoint_dir"], "last.pt")
if os.path.exists(P1_LAST) and not os.path.exists(P1_BEST):
    !python -m src.train --config {colab_cfg} --resume {P1_LAST}
else:
    print("P1 best.pt:", os.path.exists(P1_BEST), "| last.pt:", os.path.exists(P1_LAST))

---
## Phase 2 — LoRA Fine-tune MERT (`r=32, q/v/k_proj`)

In [ ]:
print("P1 best.pt:", os.path.exists(P1_BEST))
assert os.path.exists(P1_BEST), "Phase 1 chưa xong hoặc best.pt chưa sync về Drive!"
!python -m src.train --config {colab_p2_cfg} --phase1_ckpt {P1_BEST}

In [ ]:
# Resume Phase 2 nếu bị disconnect
P2_LAST = os.path.join(cfg_p2["checkpoint_dir"], "last.pt")
if os.path.exists(P2_LAST) and not os.path.exists(P2_BEST):
    !python -m src.train --config {colab_p2_cfg} --resume {P2_LAST}
else:
    print("P2 best.pt:", os.path.exists(P2_BEST), "| last.pt:", os.path.exists(P2_LAST))

---
## Export ONNX & Benchmark

In [ ]:
print("P2 best.pt:", os.path.exists(P2_BEST))
assert os.path.exists(P2_BEST), "Phase 2 chưa xong!"
!python -m src.export_onnx --config {colab_p2_cfg} --checkpoint {P2_BEST} --output {OUT_ONNX} --benchmark
print("ONNX saved to:", OUT_ONNX)

---
## Full Benchmark trên Tập `test` (647 bài, Tiled 100% thời lượng)

In [ ]:
!python -m src.evaluate --config {colab_p2_cfg} --checkpoint {P2_BEST} --split test --output {OUT_EVAL_JSON}

---
## Quick Inference Test

In [ ]:
test_audio_dir = os.path.join(DATA_DST, "test", "audio")
test_files = sorted(os.listdir(test_audio_dir))[:3]
for f in test_files:
    audio_path = os.path.join(test_audio_dir, f)
    !python -m src.infer --config {colab_p2_cfg} --checkpoint {P2_BEST} --audio {audio_path}